In [5]:
import tensorflow as tf
print(tf.__version__)
print(f"Keras: {tf.keras.__version__}")

2.19.0
Keras: 3.10.0


In [6]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [8]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
DATA_DIR ="C:\Users\Praneeth Samineni\Downloads\crop2.0final\server\newplantvillage\datasets"


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3650046776.py, line 3)

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/Users/praneeth/Documents/newplantvillage_data'

In [ ]:
class_names = list(train_gen.class_indices.keys())
print(class_names)


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input

base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = Input(shape=(128, 128, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(train_gen.num_classes, activation='softmax')(x)

model = Model(inputs, outputs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)


In [ ]:
model.save('plant_disease_model_version2.h5')
print("Model saved as plant_disease_model_version2.h5")


In [ ]:
model.save("plant_disease_model_version2.keras")
print("Model saved as plant_disease_model_version2.keras")


In [ ]:
%pip install matplotlib
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
%pip install scikit-learn
from sklearn.metrics import classification_report
import numpy as np


In [ ]:
# Reset the generator before predicting
val_gen.reset()

# Get predictions
pred_probs = model.predict(val_gen, verbose=1)

# Convert predicted probabilities to class indices
y_pred = np.argmax(pred_probs, axis=1)


In [ ]:
y_true = val_gen.classes


In [ ]:
class_labels = list(val_gen.class_indices.keys())


In [ ]:
report = classification_report(y_true, y_pred, target_names=class_labels)
print(report)


In [ ]:
# Confirm class indices match
print(val_gen.class_indices)
print(class_labels)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image


In [ ]:
img_path = '/Users/praneeth/Downloads/download (2).jpeg'
img = image.load_img(img_path, target_size=IMG_SIZE)  # Same as used in training, e.g., (128, 128)

img_array = image.img_to_array(img)
img_array = img_array / 255.0  # match rescale from ImageDataGenerator
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension


In [ ]:
pred = model.predict(img_array)
predicted_class = class_labels[np.argmax(pred)]
confidence = np.max(pred)


In [ ]:
print(f"Predicted Class: {predicted_class}")
print(f"Confidence: {confidence:.2f}")


In [ ]:
# Add this to your training script where you save the model

# Method 1: Save in multiple compatible formats
print("💾 Saving model in compatible formats...")

# Save in TensorFlow SavedModel format (most compatible)
model.save('AI_Model/plant_disease_model_savedmodel', save_format='tf')
print("✅ Saved as SavedModel format")

# Save in H5 format with specific options
model.save('AI_Model/plant_disease_model_v3.h5', save_format='h5', include_optimizer=False)
print("✅ Saved as H5 format")

# Save weights separately (most reliable)
model.save_weights('AI_Model/plant_disease_model_weights.h5')
print("✅ Saved weights separately")

# Save model architecture as JSON
model_json = model.to_json()
with open('AI_Model/plant_disease_model_architecture.json', 'w') as json_file:
    json_file.write(model_json)
print("✅ Saved model architecture")

# Method 2: Save with compatibility options for newer TensorFlow
try:
    # This should work with TensorFlow 2.19+
    model.save('AI_Model/plant_disease_model_v3.keras', save_format='keras_v3')
    print("✅ Saved in Keras v3 format")
except:
    # Fallback
    model.save('AI_Model/plant_disease_model_v3_backup.h5')
    print("✅ Saved as backup H5")

print("🎉 Model saved in multiple formats for maximum compatibility!")

# Test loading immediately after saving
print("\n🔍 Testing model loading...")
try:
    test_model = tf.keras.models.load_model('AI_Model/plant_disease_model_v3.h5')
    print("✅ H5 model loads successfully")
except Exception as e:
    print(f"❌ H5 loading failed: {e}")

try:
    test_model = tf.keras.models.load_model('AI_Model/plant_disease_model_savedmodel')
    print("✅ SavedModel loads successfully")
except Exception as e:
    print(f"❌ SavedModel loading failed: {e}")